### Basic Extraction
3/9/26

Description: 
- Explore 3 libraries, PyPDF, PDFPlumber, and pymuPDF to compare extraction capabilities

Data sources:
- https://www.usgs.gov/human-capital/nationwide-standard-position-descriptions
- https://www.commerce.gov/hr/practitioners/position-management/descriptions

In [1]:
#import needed libraries
import pandas as pd
import numpy as np
import openpyxl
import os
from openpyxl import load_workbook
import pdfplumber
import re


In [2]:
#import libraries for alternate library pypdf
import pypdf
from pypdf import PdfReader

In [3]:
#import PyMuPDF
import pymupdf

In [3]:
#pdfplumber
# PDFplumber: extract text between "major duties" and "factor 1" section headers (headers lowercased)


def extract_major_duties_to_factor1_pdfplumber(pdf_path: str) -> str | None:
    """
    Extract the section between the header containing "major duties" and the header
    containing "factor 1". Returns the section with both headers lowercased.
    """
    with pdfplumber.open(pdf_path) as pdf:
        full_text_parts = []
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    if not full_text.strip():
        return None

    lines = full_text.split("\n")
    start_idx = None
    end_idx = None

    for i, line in enumerate(lines):
        if "MAJOR DUTIES" in line:
            start_idx = i
            break
    if start_idx is None:
        return None

    for i in range(start_idx + 1, len(lines)):
        if "Factor 1:" in lines[i]:
            end_idx = i
            break
    if end_idx is None:
        # No "factor 1" found: take from start to end of document
        end_idx = len(lines)

    # Section: from start header (inclusive) to just before "factor 1" line
    section_lines = lines[start_idx:end_idx]

    result = "\n".join(section_lines).strip()
    return result if result else None


# Example usage:
# result = extract_major_duties_to_factor1_pdfplumber("Accountant 05 (1).pdf")
# print(result)

In [4]:
#test pdfplumber code
txt1 = extract_major_duties_to_factor1_pdfplumber("Accountant 05 (1).pdf")
txt1

'II. MAJOR DUTIES AND RESPONSIBILITIES\nPerforms a variety of routine technical accounting assignments that are structured to provide the incumbent with\nexperience in the application of accounting principles, procedures, and techniques. Duties typically performed\ninclude the following: examining accounting documents for proper accounting classification and authorization;\nperforming reconciliations; analyzing a variety of accounts; entering and processing data into various accounts\nand the general ledger; recognizing and adjusting differences between the general ledger and subsidiary\naccounts; preparing monthly trial balances and financial reports; reviewing procedures related to the automated\naccounting systems; reviewing, for completeness, financial data submitted by business firms.\nIII. FACTOR LEVELS\nFactor 1 \xad Knowledge Required by the Position FL 1\xad5, 750 points\nProfessional knowledge of the concepts and principles of accounting needed to perform developmental\nassig

In [5]:
#batch test code on group of PDFs of similar type

data_dict = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_major_duties_to_factor1_pdfplumber(f"HR/{file}")
    if df_row:  # Only append non-empty results
        data_dict[f"{file}"] = df_row

# #region agent log
import json
with open("debug-f1cfa6.log", "a") as _f:
    _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict), "value_types": [type(v).__name__ for v in (list(data_dict.values())[:3] if data_dict else [])], "sample_key": list(data_dict.keys())[0] if data_dict else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# #endregion

HR_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(HR_df)
HR_df.to_excel("HR_pdfplumber.xlsx")

,file_name,duties
0,GS-0201-05_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...
1,GS-0201-07_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...
2,GS-0201-09_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...
3,GS-0201-11_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...
4,GS-0201-12_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...
5,GS-0201-13_HRSpecialist.pdf,MAJOR DUTIES (indicate percentages of time equ...


In [ ]:
data_dict2 = {}  # list to hold each row dict

for i, file in enumerate(os.listdir("Mixed_series_PDF")):
    df_row = extract_major_duties_to_factor1_pdfplumber(f"Mixed_series_PDF/{file}")
    if df_row:  # Only append non-empty results
        data_dict2[f"{file}"] = df_row

# #region agent log
import json
with open("debug-f1cfa6.log", "a") as _f:
    _f.write(json.dumps({"sessionId": "f1cfa6", "runId": "pre-fix", "hypothesisId": "H1", "location": "job_description_analysis_m2.ipynb:batch cell", "message": "data_dict before DataFrame", "data": {"len": len(data_dict2), "value_types": [type(v).__name__ for v in (list(data_dict2.values())[:3] if data_dict2 else [])], "sample_key": list(data_dict2.keys())[0] if data_dict2 else None}, "timestamp": __import__("time").time() * 1000}) + "\n")
# #endregion

mixed_series_df = pd.DataFrame([{"file_name": k, "duties": v} for k, v in data_dict2.items()])
# test_df.reset_index(inplace=True, columns = ['file_name', 'duties'])  # removed: reset_index has no 'columns' param; columns already set above
#test_df.head()
#test_df.shape
display(mixed_series_df)
mixed_series_df.to_excel("Mixed_series_pdfplumber.xlsx")

position-attorney-13-type-2-level-d.pdf
position-attorney-15-type-3-level-e.pdf
position-business-and-industry-assistant-05.pdf
position-human-resources-specialist-05.pdf
position-human-resources-specialist-14.pdf
position-information-technology-specialist-13.pdf


,file_name,duties
0,position-attorney-13-type-2-level-d.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\nPerform...
1,position-attorney-15-type-3-level-e.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\nAs the ...
2,position-business-and-industry-assistant-05.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\nProvide...
3,position-human-resources-specialist-05.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\nAssignm...
4,position-human-resources-specialist-14.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\nHas ful...
5,position-information-technology-specialist-13.pdf,II. MAJOR DUTIES AND RESPONSIBILITIES\n-- Anal...


In [5]:
#trying with PDF plumber shows boxes and extracts that don't make sense
#how do we turn the boxes into columns? The quality of the extract is pretty good
#note: explored metadata, but this did not help. Wanted to see if we can get a sense of the parameters to locate the major duties section
with pdfplumber.open(f"HR/GS-0201-07_HRSpecialist.pdf") as pdf2:
    first_page = pdf2.pages[1]
    #print(first_page.chars[0])
    first_page_text = first_page.extract_text(layout= False)
    #explore parameters
    first_page_words = first_page.extract_words()
    display(first_page_words)
    #print(pdf2.metadata)
    

[{'text': 'DOI',
  'x0': 259.73,
  'x1': 282.398,
  'top': 73.75199999999995,
  'doctop': 865.752,
  'bottom': 85.75199999999995,
  'upright': True,
  'height': 12.0,
  'width': 22.668000000000006,
  'direction': 'ltr'},
 {'text': 'Standard',
  'x0': 285.398,
  'x1': 333.4580000000001,
  'top': 73.75199999999995,
  'doctop': 865.752,
  'bottom': 85.75199999999995,
  'upright': True,
  'height': 12.0,
  'width': 48.06000000000006,
  'direction': 'ltr'},
 {'text': 'PD',
  'x0': 336.494,
  'x1': 352.49,
  'top': 73.75199999999995,
  'doctop': 865.752,
  'bottom': 85.75199999999995,
  'upright': True,
  'height': 12.0,
  'width': 15.995999999999981,
  'direction': 'ltr'},
 {'text': 'PD#',
  'x0': 269.81,
  'x1': 291.77,
  'top': 89.61199999999997,
  'doctop': 881.612,
  'bottom': 101.61199999999997,
  'upright': True,
  'height': 12.0,
  'width': 21.95999999999998,
  'direction': 'ltr'},
 {'text': 'DC00600',
  'x0': 294.77,
  'x1': 342.05,
  'top': 89.61199999999997,
  'doctop': 881.612,
 

In [6]:
#check metadata
with pdfplumber.open("Accountant 05 (1).pdf") as pdf3:
    first_page = pdf3.pages[0]
    #print(first_page.chars[0])
    first_page_text = first_page.extract_text()
    print(first_page_text)
    #print(pdf3.extract_text_lines())

2/8/2019 Accountant 05 - OHRM
Home > HR Practitioners > Classification & Position Management > PD Library
Accountant 05
GS­0510­05
NOTE: THE SENTENCE IN PART I DESCRIBING THE PURPOSE OF THE POSITION AND PARTS II AND III IN THEIR
ENTIRETY ARE PERMANENT PARTS OF THE LIBRARY AND MAY NOT BE CHANGED OR EDITED IN ANY WAY.
I. INTRODUCTION
This position is located in
The incumbent of this position serves as a trainee accountant, utilizing a professional knowledge of accounting
principles and procedures in carrying out developmental assignments.
II. MAJOR DUTIES AND RESPONSIBILITIES
Performs a variety of routine technical accounting assignments that are structured to provide the incumbent with
experience in the application of accounting principles, procedures, and techniques. Duties typically performed
include the following: examining accounting documents for proper accounting classification and authorization;
performing reconciliations; analyzing a variety of accounts; entering and processing 

## Major Duties Section Extraction

Extract the "Major duties" (or "Primary responsibilities") section from position description PDFs using three libraries: **pdfplumber**, **pypdf**, and **PyMuPDF**. The section is bounded by headers like "II. MAJOR DUTIES AND RESPONSIBILITIES" and "III. FACTOR LEVELS".

In [4]:
# pip install pymupdf  # Uncomment if not installed
import re
import fitz  # PyMuPDF

In [ ]:
# Section markers - configurable for different PD formats
START_PATTERNS = [
    r"II\.\s*MAJOR DUTIES AND RESPONSIBILITIES",
    r"MAJOR DUTIES AND RESPONSIBILITIES",
    r"Major duties and responsibilities",
    r"Major duties",
    r"Primary responsibilities",
]
END_PATTERNS = [
    r"III\.\s*FACTOR LEVELS",
    r"III\.\s*FACTOR",
    r"FACTOR LEVELS",
    r"Factor 1\s*[–\-]\s*Knowledge",  # "Factor 1 – Knowledge"
]


def extract_major_duties_pdfplumber(pdf_path: str) -> str | None:
    """Extract Major duties section using pdfplumber."""
    full_text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    return _extract_section_by_patterns(full_text)


def extract_major_duties_pypdf(pdf_path: str) -> str | None:
    """Extract Major duties section using pypdf."""
    reader = PdfReader(pdf_path)
    full_text_parts = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text_parts.append(text)
    full_text = "\n".join(full_text_parts)
    print(full_text)
    return _extract_section_by_patterns(full_text)


def extract_major_duties_pymupdf(pdf_path: str) -> str | None:
    """Extract Major duties section using PyMuPDF (fitz)."""
    doc = fitz.open(pdf_path)
    full_text_parts = []
    for page in doc:
        full_text_parts.append(page.get_text())
    doc.close()
    full_text = "\n".join(full_text_parts)
    print(full_text)
    return _extract_section_by_patterns(full_text)


def _extract_section_by_patterns(text: str) -> str | None:
    """Find text between start and end patterns (shared logic)."""
    start_match = None
    for pat in START_PATTERNS:
        start_match = re.search(pat, text, re.IGNORECASE)
        if start_match:
            print({pat}"is a match")
            break
    if not start_match:
        return None
    # Start after the header line
    search_from = start_match.end()
    # Find end - look in text after start
    remainder = text[search_from:]
    end_match = None
    for pat in END_PATTERNS:
        end_match = re.search(pat, remainder, re.IGNORECASE)
        if end_match:
            break
    if end_match:
        section_text = remainder[: end_match.start()].strip()
    else:
        section_text = remainder.strip()
    return section_text if section_text else None

#3/24/26 Look back to understand by the other 2 are not working

In [8]:
# Demo: Extract Major duties from a sample PDF using all three libraries
SAMPLE_PDF = "Accountant 05 (1).pdf"  # or "HR/GS-0201-07_HRSpecialist.pdf"

print("=== pdfplumber ===")
result_plumber = extract_major_duties_pdfplumber(SAMPLE_PDF)
print(result_plumber[:500] + "..." if result_plumber and len(result_plumber) > 500 else result_plumber)

print("\n=== pypdf ===")
result_pypdf = extract_major_duties_pypdf(SAMPLE_PDF)
print(result_pypdf[:500] + "..." if result_pypdf and len(result_pypdf) > 500 else result_pypdf)

print("\n=== PyMuPDF ===")
result_pymupdf = extract_major_duties_pymupdf(SAMPLE_PDF)
print(result_pymupdf[:500] + "..." if result_pymupdf and len(result_pymupdf) > 500 else result_pymupdf)

=== pdfplumber ===
Performs a variety of routine technical accounting assignments that are structured to provide the incumbent with
experience in the application of accounting principles, procedures, and techniques. Duties typically performed
include the following: examining accounting documents for proper accounting classification and authorization;
performing reconciliations; analyzing a variety of accounts; entering and processing data into various accounts
and the general ledger; recognizing and adjusting diff...

=== pypdf ===
2/8/2019 Accountant 05 - OHRM
https://hr.commerce.gov/Practitioners/ClassiﬁcationAndPositionManagement/PDLibrary/prod01_000377?format_for_print=true 1/2
 
    
Home > HR Practitioners > Classification & Position Management > PD Library
Accountant 05
GS­0510­05  
NOTE: THE SENTENCE IN PART I DESCRIBING THE PURPOSE OF THE POSITION AND PARTS II AND III IN THEIR
ENTIRETY ARE PERMANENT PARTS OF THE LIBRARY AND MAY NOT BE CHANGED OR EDITED IN ANY WAY. 
I. INTRODUC

## pypdf

In [ ]:
#pip install pypdf

In [7]:
#3/3/26 used dummy code from google AI search
#extracts data on cover page only

def extract_form_data(pdf_path):
    reader = PdfReader(pdf_path)
    form_data = {}
    
    # Iterate over pages and get fields
    for page in reader.pages[0]:
        # get_fields() returns a dictionary of field data
        fields = reader.get_fields()
        if fields:
            for field_name, field_data in fields.items():
                # Extract the value (/V) from the field data
                form_data[field_name] = field_data.get('/V', '')
                
    return form_data



In [8]:
#display
pd.set_option('display.max_rows', None, 'display.max_columns', None)

#execute function
test_dict = extract_form_data(f"HR/GS-0201-07_HRSpecialist.pdf")
test_df = pd.DataFrame(test_dict, index=[0])
test_df.head()

,1 Agency Position No,2ReasonForSubmission,2ReasonForSubmissionExplanation,3Service,4 Employing Office Location,5 Duty Station,6 OPM Certification No,7FLSA,8FinancialStatementExecutive,8FinancialStatementEmployment,9SubjectToIAaction,10PositionStatus,11PositionIs,11PositionIs1st,12SensitivityNonSensitiveADP,12Sensitivity,12SensitivityNonCriticalSensitiveADP,12SensitivityCriticalADP,12SensitivitySpecialADP,13 Competitive Level Code,14 Agency Use,15Atitle,15ApayPlan,15AoccupationalCode,15Agrade,15Ainitials,15Adate,15Btitle,15BpayPlan,15BoccupationalCode,15Bgrade,15bInitials,15Bdate,15Ctitle,15CpayPlan,15CoccupationalCode,15Cgrade,15Cinitials,15Cdate,15Dtitle,15DpayPlan,15DoccupationalCode,15Dgrade,15Dinitials,15Ddate,15Etitle,15EpayPlan,15EoccupationalCode,15Egrade,15Einitials,15Edate,16OrgTitle,17NameOfEmployee,18 Department Agency or Establishment,18AfirstSubdivision,18b Second Subdivision,18c Third Subdivision,18d Fourth Subdivision,18e Fifth Subdivision,19SignatureEmployee,20a Typed Name and Title of Immediate Supervisor,20aSignatureImmediateSupervisor,20aDate,20b Typed Name and Title of HigherLevel Supervisor or Manager optional,20bSignatureHigher,20bDate,21Typed Name and Title of Official Taking Action,21SignatureOfficialTakingAction,21Date,22 Position Classification Standards Used in ClassifyingGrading Position,23aEmployeeInitials1,23aEmployeeDate1,23aEmployeeInitials2,23aEmployeeDate2,23aEmployeeInitials3,23aEmployeeDate3,23aEmployeeInitials4,23aEmployeeDate4,23aEmployeeInitials5,23aEmployeeDate5,23bSupervisorInitials1,23bSupervisorDate1,23bSupervisorInitials2,23bSupervisorDate2,23bSupervisorInitials3,23bSupervisorDate3,23bSupervisorInitials4,23bSupervisorDate4,23bSupervisorInitials5,23bSupervisorDate5,23cClassifierInitials1,23cClassifierDate2,23cClassifierInitials2,23cClassifierDate,23cClassifierInitials3,23cClassifierDate3,23cClassifierInitials4,23cClassifierDate4,23cClassifierInitials5,23cClassifierDate5,24Remarks
0,DC00600,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,7,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,NaN,06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Developmental position


In [10]:
for file in os.listdir("HR"):
    print(file)

GS-0201-05_HRSpecialist.pdf
GS-0201-07_HRSpecialist.pdf
GS-0201-09_HRSpecialist.pdf
GS-0201-11_HRSpecialist.pdf
GS-0201-12_HRSpecialist.pdf
GS-0201-13_HRSpecialist.pdf


In [ ]:
#test running on a several PDs

# Improved approach: build up a list of dicts, then build DataFrame once at end,
# and make sure the dataframe is being expanded, instead of unused 'pdf_df'

rows = []  # list to hold each row dict

for i, file in enumerate(os.listdir("HR")):
    df_row = extract_form_data(f"HR/{file}")
    if df_row:  # Only append non-empty results
        rows.append(df_row)

test_df = pd.DataFrame(rows)
test_df.reset_index(drop=True, inplace=True)
#test_df.head()
#test_df.shape
display(test_df)



,1 Agency Position No,2ReasonForSubmission,2ReasonForSubmissionExplanation,3Service,4 Employing Office Location,5 Duty Station,6 OPM Certification No,7FLSA,8FinancialStatementExecutive,8FinancialStatementEmployment,9SubjectToIAaction,10PositionStatus,11PositionIs,11PositionIs1st,12SensitivityNonSensitiveADP,12Sensitivity,12SensitivityNonCriticalSensitiveADP,12SensitivityCriticalADP,12SensitivitySpecialADP,13 Competitive Level Code,14 Agency Use,15Atitle,15ApayPlan,15AoccupationalCode,15Agrade,15Ainitials,15Adate,15Btitle,15BpayPlan,15BoccupationalCode,15Bgrade,15bInitials,15Bdate,15Ctitle,15CpayPlan,15CoccupationalCode,15Cgrade,15Cinitials,15Cdate,15Dtitle,15DpayPlan,15DoccupationalCode,15Dgrade,15Dinitials,15Ddate,15Etitle,15EpayPlan,15EoccupationalCode,15Egrade,15Einitials,15Edate,16OrgTitle,17NameOfEmployee,18 Department Agency or Establishment,18AfirstSubdivision,18b Second Subdivision,18c Third Subdivision,18d Fourth Subdivision,18e Fifth Subdivision,19SignatureEmployee,20a Typed Name and Title of Immediate Supervisor,20aSignatureImmediateSupervisor,20aDate,20b Typed Name and Title of HigherLevel Supervisor or Manager optional,20bSignatureHigher,20bDate,21Typed Name and Title of Official Taking Action,21SignatureOfficialTakingAction,21Date,22 Position Classification Standards Used in ClassifyingGrading Position,23aEmployeeInitials1,23aEmployeeDate1,23aEmployeeInitials2,23aEmployeeDate2,23aEmployeeInitials3,23aEmployeeDate3,23aEmployeeInitials4,23aEmployeeDate4,23aEmployeeInitials5,23aEmployeeDate5,23bSupervisorInitials1,23bSupervisorDate1,23bSupervisorInitials2,23bSupervisorDate2,23bSupervisorInitials3,23bSupervisorDate3,23bSupervisorInitials4,23bSupervisorDate4,23bSupervisorInitials5,23bSupervisorDate5,23cClassifierInitials1,23cClassifierDate2,23cClassifierInitials2,23cClassifierDate,23cClassifierInitials3,23cClassifierDate3,23cClassifierInitials4,23cClassifierDate4,23cClassifierInitials5,23cClassifierDate5,24Remarks
0,DC00500,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,5,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65282, 104688, 122191], '/C...",06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Developmental position
1,DC00600,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,7,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65262, 104668, 185509], '/C...",06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Developmental position
2,DC00700,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,9,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65147, 104553, 159998], '/C...",06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,DC00800,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,11,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65181, 104587, 166318], '/C...",06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,DC00900,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,12,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65180, 104586, 196711], '/C...",06/22/2020,Administrative Work in the Human Resources Man...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5,DC01000,/Other,New DOI SPD,,,,,,,,/Yes,,,,,,,,,,,,,,,,,HR Specialist,GS,201,13,rl,06/22/2020,,,,,,,,,,,,,,,,,,,,,Department of the Interior,,,,,,,,,,,,,Renae Lockwood Classification Program Manager,"{'/ByteRange': [0, 65165, 104571, 194809], '/C...",06/22/2020,Administrative Work

In [ ]:
#export to excel
test_df.to_excel("test_df.xlsx", index=False)